# Step 7 - Scores on all perturbed graphs

For every setting (3 kinds x 3 levels) and every run (K = 30):
perturb -> recompute the 5 features -> **frozen** scaler -> both forests **without retraining**.
The single centralities are kept as computed. Logic is in `src/experiment.py`.

Scores are saved in natural units (decision D12): raw values for the single centralities, PageRank units for the forests.

The full run takes ~13 minutes. Settings already saved in `results/runs/` are skipped, so the notebook can be stopped and restarted. The raw runs (~150 MB) are not on GitHub - only the result tables are.

In [1]:
import os, sys
if 'google.colab' in sys.modules and not os.path.exists('Final-Project'):
    !git clone https://github.com/ShaharBar99/Final-Project.git
    %cd Final-Project
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

## 1. Load the frozen pieces from steps 3-4

In [2]:
import joblib
import pandas as pd
from src import analysis, config, data, experiment, model

g, nodes = data.load_clean_graph()
scaler = joblib.load(config.PROCESSED_DIR / 'scaler.joblib')
x = pd.read_csv(config.PROCESSED_DIR / 'features_scaled.csv', index_col='paper_id')
forests = {name: model.load_or_train(name, x[cols], x[model.TARGET], config.PROCESSED_DIR / 'models')
           for name, cols in experiment.VARIANTS.items()}

## 2. Check: the pipeline reproduces the base ranking of step 5

Scoring the *original* graph through the step 7 code must give exactly the rankings saved in step 5.

In [3]:
from src import ranking
scores, out_of_range = experiment.score_graph(g, scaler, forests)
old = pd.read_csv(config.PROCESSED_DIR / 'base_ranking.csv', index_col='paper_id', dtype={'paper_id': str})
same = {m: (old.loc[scores.index, m] == ranking.to_ranking(scores[m])).all() for m in scores}
print(same)
assert all(same.values())

{'rf_with_pagerank': np.True_, 'rf_without_pagerank': np.True_, 'degree': np.True_, 'betweenness': np.True_, 'closeness': np.True_, 'pagerank': np.True_, 'kcore': np.True_}


## 3. Run everything (K = 30)

In [4]:
experiment.run_all(g, scaler, forests, k=30)

remove_05: already done, skipped
remove_10: already done, skipped
remove_20: already done, skipped
add_random_05: already done, skipped
add_random_10: already done, skipped
add_random_20: already done, skipped
add_dag_05: already done, skipped
add_dag_10: already done, skipped
add_dag_20: already done, skipped


## 4. What happened to the inputs of the forest?

Share of papers whose scaled feature fell **outside [0, 1]** - values the forest never saw in training. A tree cannot extrapolate: for such values it returns the same output as at the edge of the range. This is the saturation effect we said we would watch for.

In [5]:
oor = analysis.out_of_range_table()
oor.round(4)

,kind,level,degree,betweenness,closeness,pagerank,kcore,seconds_per_run
0,remove,0.05,0.0233,0.0000,0.0000,0.0043,0.0233,0.4913
1,remove,0.10,0.0478,0.0000,0.0000,0.0000,0.0478,0.4493
2,remove,0.20,0.0999,0.0000,0.0000,0.0000,0.0999,0.4723
3,add_random,0.05,0.0000,0.0100,0.0013,0.4575,0.0000,0.8433
4,add_random,0.10,0.0000,0.0334,0.0588,0.5794,0.0000,2.3510
5,add_random,0.20,0.0000,0.0951,0.6003,0.5836,0.0000,5.7983
6,add_dag,0.05,0.0000,0.0001,0.0003,0.4221,0.0000,0.5017
7,add_dag,0.10,0.0000,0.0001,0.0017,0.5667,0.0000,0.5387
8,add_dag,0.20,0.0000,0.0006,0.0083,0.6038,0.0000,0.6307
